# Membro 1 — Integração com a API Olho Vivo (Infraestrutura)

Requisitos **02** (integração com a API) e **07** (falhas da API).

Este é o notebook de rascunho do Membro 1. As seções 1 e 2 seguem a estrutura do notebook final; a seção 1 (Domínio) é **provisória** e será substituída pelo código do Membro 2.

| Atividade | Onde |
|---|---|
| 1. Cadastro, token, `.env` e `.gitignore` | Seção 0 |
| 2. Mapa de endpoints | Seção 0.1 |
| 3. `OlhoVivoClient` (Session, timeout, reautenticação) | Seção 2 |
| 4. Tratamento de falhas | Seções 1 e 2 (testes na 2.1) |
| 5. JSON reais para os colegas | Seção 2.2 |

## 0. Configuração (atividade 1)

**Passo a passo manual (uma vez):**
1. Criar conta no portal de desenvolvedores da SPTrans e confirmar o e-mail.
2. Criar uma aplicação e copiar o token gerado.
3. Na raiz do repositório, copiar `.env.example` para `.env` e colar o token em `OLHOVIVO_TOKEN`.
4. Conferir que o `.gitignore` contém `.env` **antes do primeiro commit** (`git status` não pode listar o `.env`).

Se o token vazar num commit, gere outro no portal: apagar o commit não basta.

In [ ]:
import json
import os
from dataclasses import dataclass
from http import HTTPStatus
from pathlib import Path
from types import SimpleNamespace
from unittest.mock import Mock

import requests
from dotenv import load_dotenv

load_dotenv()
TOKEN = os.getenv("OLHOVIVO_TOKEN")  # nunca escrever o token no código

### 0.1 Endpoints mapeados (atividade 2)

Base: `http://api.olhovivo.sptrans.com.br/v2.1` (conferir na documentação oficial se mudou).

| Uso | Método | Endpoint | Parâmetros | Resposta (campos principais) |
|---|---|---|---|---|
| Autenticação | POST | `/Login/Autenticar` | `token` | `true`/`false`; guarda cookie de sessão |
| Buscar linhas | GET | `/Linha/Buscar` | `termosBusca` | lista: `cl` código, `lt` letreiro, `tl` sufixo, `sl` sentido (1 ou 2), `tp`/`ts` terminais |
| Paradas por linha | GET | `/Parada/BuscarParadasPorLinha` | `codigoLinha` | lista: `cp` código, `np` nome, `ed` endereço, `py` lat, `px` lon |
| Posição dos veículos | GET | `/Posicao/Linha` | `codigoLinha` | `hr` hora, `vs`: lista com `p` prefixo, `py` lat, `px` lon |
| Posição de todos | GET | `/Posicao` | — | `hr`, `l`: linhas, cada uma com `vs` (pode ser grande) |
| Previsão | GET | `/Previsao` | `codigoParada`, `codigoLinha` | `hr`, `p`: parada com `l` (linhas) e `vs` (veículos com `t` horário previsto) |

Observações:
- A autenticação vale por cookie; a sessão expira e a API passa a responder **401**.
- Não existe busca de paradas por coordenada (isso é desafio do Membro 2).
- `sl` = 1 é terminal principal → secundário; `sl` = 2 é o inverso. Cada linha aparece uma vez por sentido.

## 1. Domínio — provisório (será substituído pelo Membro 2)

Só o mínimo que o cliente precisa para devolver entidades e lançar exceções. Os campos devem ser combinados na reunião inicial.

In [ ]:
class OlhoVivoError(Exception):
    """Base de todos os erros da integração."""

class AutenticacaoError(OlhoVivoError):
    """Token ausente, recusado ou sessão não renovável."""

class ApiIndisponivelError(OlhoVivoError):
    """Timeout, falha de rede ou erro 5xx."""

class RespostaInvalidaError(OlhoVivoError):
    """Resposta vazia, não-JSON ou com campos inesperados."""


@dataclass(frozen=True)
class Coordenada:
    """Ponto geográfico em graus decimais."""
    latitude: float
    longitude: float

@dataclass(frozen=True)
class Parada:
    """Ponto de parada de ônibus."""
    codigo: int
    nome: str
    endereco: str
    coordenada: Coordenada

@dataclass(frozen=True)
class Linha:
    """Linha de ônibus em um sentido (cada linha aparece uma vez por sentido)."""
    codigo: int
    letreiro: str   # ex.: "8000-10"
    sentido: int    # 1 ou 2
    origem: str
    destino: str

@dataclass(frozen=True)
class Previsao:
    """Previsão de chegada de uma linha em uma parada."""
    codigo_parada: int
    codigo_linha: int
    consultada_em: str      # "HH:MM", hora da API
    chegadas: tuple[str, ...]  # horários previstos "HH:MM"; vazio = sem previsão

## 2. Infraestrutura — `OlhoVivoClient` (atividades 3 e 4)

- `requests.Session` mantém o cookie de autenticação.
- Todo acesso tem `timeout`.
- Resposta **401** dispara uma nova autenticação e **uma** nova tentativa.
- Toda falha vira uma exceção própria; o resto do sistema nunca vê `requests` nem JSON bruto.
- Lista vazia de linhas/paradas é resposta válida (quem decide o que fazer é o caso de uso). Já a previsão sem veículos vira `Previsao` com `chegadas` vazio.

In [ ]:
BASE_URL = "http://api.olhovivo.sptrans.com.br/v2.1"
TIMEOUT_S = 10


class OlhoVivoClient:
    """Cliente da API Olho Vivo: sessão autenticada, timeout e erros próprios."""

    def __init__(self, token, session=None, timeout=TIMEOUT_S):
        if not token:
            raise AutenticacaoError("Token ausente: defina OLHOVIVO_TOKEN no .env.")
        self._token = token
        self._session = session or requests.Session()
        self._timeout = timeout
        self._autenticado = False

    # --- acesso HTTP ---------------------------------------------------
    def _requisitar(self, metodo, caminho, **params):
        try:
            resp = self._session.request(
                metodo, BASE_URL + caminho, params=params, timeout=self._timeout
            )
        except requests.Timeout as e:
            raise ApiIndisponivelError("Tempo esgotado ao consultar a API.") from e
        except requests.RequestException as e:
            raise ApiIndisponivelError(f"Falha de comunicação com a API: {e}") from e
        if resp.status_code >= HTTPStatus.INTERNAL_SERVER_ERROR:
            raise ApiIndisponivelError(f"API respondeu HTTP {resp.status_code}.")
        return resp

    def autenticar(self):
        """Autentica e guarda o cookie de sessão."""
        resp = self._requisitar("POST", "/Login/Autenticar", token=self._token)
        if resp.text.strip().lower() != "true":
            raise AutenticacaoError("Token recusado pela API.")
        self._autenticado = True

    def get_json(self, caminho, **params):
        """GET autenticado; devolve o JSON bruto (usado também para salvar amostras)."""
        if not self._autenticado:
            self.autenticar()
        resp = self._requisitar("GET", caminho, **params)
        if resp.status_code == HTTPStatus.UNAUTHORIZED:  # sessão expirou
            self.autenticar()
            resp = self._requisitar("GET", caminho, **params)
            if resp.status_code == HTTPStatus.UNAUTHORIZED:
                raise AutenticacaoError("Não foi possível renovar a sessão.")
        if not resp.ok:
            raise RespostaInvalidaError(f"HTTP {resp.status_code} em {caminho}.")
        try:
            return resp.json()
        except ValueError as e:
            raise RespostaInvalidaError(f"Resposta vazia ou inválida em {caminho}.") from e

    # --- conversão JSON -> entidades -----------------------------------
    @staticmethod
    def _converter(dados, fn, caminho):
        try:
            return fn(dados)
        except (AttributeError, KeyError, TypeError, ValueError) as e:
            raise RespostaInvalidaError(f"Dados inconsistentes em {caminho}: {e!r}") from e

    def buscar_linhas(self, termo):
        """Busca linhas pelo termo (número ou nome)."""
        caminho = "/Linha/Buscar"
        dados = self.get_json(caminho, termosBusca=termo)

        def converter(lista):
            linhas = []
            for i in lista:
                sentido = int(i["sl"])
                principal, secundario = i["tp"], i["ts"]
                ida = sentido == 1
                origem, destino = (principal, secundario) if ida else (secundario, principal)
                linhas.append(Linha(int(i["cl"]), f'{i["lt"]}-{i["tl"]}', sentido, origem, destino))
            return linhas
        return self._converter(dados, converter, caminho)

    def paradas_da_linha(self, codigo_linha):
        """Lista as paradas atendidas por uma linha."""
        caminho = "/Parada/BuscarParadasPorLinha"
        dados = self.get_json(caminho, codigoLinha=codigo_linha)

        def converter(lista):
            return [
                Parada(
                    int(p["cp"]),
                    p["np"],
                    p.get("ed", ""),
                    Coordenada(float(p["py"]), float(p["px"])),
                )
                for p in lista
            ]
        return self._converter(dados, converter, caminho)

    def posicoes_veiculos(self, codigo_linha):
        """Posição atual dos veículos de uma linha."""
        caminho = "/Posicao/Linha"
        dados = self.get_json(caminho, codigoLinha=codigo_linha)
        return self._converter(
            dados,
            lambda d: [Coordenada(float(v["py"]), float(v["px"])) for v in d["vs"]],
            caminho,
        )

    def previsao(self, codigo_parada, codigo_linha):
        """Horários previstos de chegada; sem veículos, `chegadas` fica vazio."""
        caminho = "/Previsao"
        dados = self.get_json(caminho, codigoParada=codigo_parada, codigoLinha=codigo_linha)

        def converter(d):
            linhas = (d.get("p") or {}).get("l") or []
            chegadas = tuple(v["t"] for lin in linhas for v in lin.get("vs") or [])
            return Previsao(codigo_parada, codigo_linha, d["hr"], chegadas)
        return self._converter(dados, converter, caminho)

### 2.1 Testes offline do tratamento de falhas (atividade 4)

Usam uma sessão falsa, então rodam sem token e sem internet. Cobrem: sucesso, sessão expirada, token recusado, timeout, erro 5xx, resposta vazia, dados inconsistentes e previsão indisponível.

In [ ]:
def _resp(status=200, corpo=None, texto=""):
    def corpo_json():
        if corpo is None:
            raise ValueError("sem corpo")
        return corpo
    return SimpleNamespace(
        status_code=status, ok=status < HTTPStatus.BAD_REQUEST, text=texto, json=corpo_json
    )


def _sessao(*respostas):
    """Sessão falsa: entrega as respostas na ordem; se for exceção, lança."""
    return Mock(request=Mock(side_effect=list(respostas)))


LOGIN_OK = _resp(texto="true")
PARADAS = [{"cp": 1, "np": "Praça", "ed": "R. A", "py": -23.5, "px": -46.6}]

def _erro(cls, *respostas, chamada=lambda c: c.paradas_da_linha(1)):
    try:
        chamada(OlhoVivoClient("t", session=_sessao(*respostas)))
    except cls:
        return True
    raise AssertionError(f"esperava {cls.__name__}")

# sucesso
c = OlhoVivoClient("t", session=_sessao(LOGIN_OK, _resp(corpo=PARADAS)))
assert c.paradas_da_linha(1)[0].coordenada == Coordenada(-23.5, -46.6)

# sessão expirada: 401 -> reautentica -> repete
c = OlhoVivoClient("t", session=_sessao(LOGIN_OK, _resp(401), LOGIN_OK, _resp(corpo=PARADAS)))
assert len(c.paradas_da_linha(1)) == 1

# falhas
assert _erro(AutenticacaoError, _resp(texto="false"))                       # token recusado
assert _erro(AutenticacaoError, LOGIN_OK, _resp(401), _resp(texto="false")) # não renova
assert _erro(ApiIndisponivelError, requests.Timeout())                      # timeout
assert _erro(ApiIndisponivelError, requests.ConnectionError())              # sem rede
assert _erro(ApiIndisponivelError, LOGIN_OK, _resp(503))                    # erro 5xx
assert _erro(RespostaInvalidaError, LOGIN_OK, _resp(200, corpo=None))       # corpo vazio
assert _erro(RespostaInvalidaError, LOGIN_OK, _resp(404))                   # 4xx
assert _erro(RespostaInvalidaError, LOGIN_OK, _resp(corpo=[{"cp": 1}]))     # campos faltando
try:                                                                        # token ausente
    OlhoVivoClient("")
except AutenticacaoError:
    pass
else:
    raise AssertionError("esperava AutenticacaoError")

# sentido: 1 = principal -> secundário; 2 = inverso
base = {"cl": 1, "lt": "8000", "tl": "10", "tp": "TERM A", "ts": "TERM B"}
sentidos = [{**base, "sl": 1}, {**base, "sl": 2}]
c = OlhoVivoClient("t", session=_sessao(LOGIN_OK, _resp(corpo=sentidos)))
trechos = [(x.origem, x.destino) for x in c.buscar_linhas("8000")]
assert trechos == [("TERM A", "TERM B"), ("TERM B", "TERM A")]

# resposta com formato inesperado (lista em vez de objeto)
assert _erro(RespostaInvalidaError, LOGIN_OK, _resp(corpo=[]), chamada=lambda c: c.previsao(1, 2))

# previsão indisponível: vira Previsao vazia, não erro
c = OlhoVivoClient("t", session=_sessao(LOGIN_OK, _resp(corpo={"hr": "10:00", "p": None})))
assert not c.previsao(1, 2).chegadas

print("Todos os testes passaram.")

### 2.2 Amostras reais em JSON para os outros membros (atividade 5)

Precisa de token válido. Salva em `dados/` as respostas **brutas** da API para uma linha de exemplo, para o grupo desenvolver sem depender da API. Depois de rodar, os JSON entram no repositório (não contêm o token).

In [ ]:
def coletar_amostras(cliente, termo_linha="8000", pasta="dados"):
    """Salva respostas brutas da API em `pasta` e devolve a lista de arquivos criados."""
    destino = Path(pasta)
    destino.mkdir(exist_ok=True)

    def salvar(nome, dados):
        caminho = destino / nome
        texto = json.dumps(dados, ensure_ascii=False, indent=2)
        caminho.write_text(texto, encoding="utf-8")
        return caminho

    linhas = cliente.get_json("/Linha/Buscar", termosBusca=termo_linha)
    if not linhas:
        raise RespostaInvalidaError(f"Nenhuma linha encontrada para '{termo_linha}'.")
    cl = linhas[0]["cl"]
    paradas = cliente.get_json("/Parada/BuscarParadasPorLinha", codigoLinha=cl)
    arquivos = [
        salvar("linhas_buscar.json", linhas),
        salvar(f"paradas_linha_{cl}.json", paradas),
        salvar(f"posicao_linha_{cl}.json", cliente.get_json("/Posicao/Linha", codigoLinha=cl)),
        salvar("posicao_todos.json", cliente.get_json("/Posicao")),
    ]
    if paradas:
        cp = paradas[0]["cp"]
        previsao = cliente.get_json("/Previsao", codigoParada=cp, codigoLinha=cl)
        arquivos.append(salvar(f"previsao_{cp}_{cl}.json", previsao))
    return arquivos

In [ ]:
# Execução com token real (rodar só depois de configurar o .env)
for arquivo in coletar_amostras(OlhoVivoClient(TOKEN)):
    print("salvo:", arquivo)